# 02 -- Data loading

Convert the Arctic Shift / Pushshift archives of r/litigi to Parquet.

The archives are streamed and written in chunks, so memory use does not depend on the archive size. Settings (paths, date range) come from `subreddit-lens.toml` next to this notebook.

Expected input files in `data/`:

- `litigi_comments.zst` (required)
- `litigi_submissions.zst` (optional: needed to count replies to the author of each post)

In [ ]:
import logging
from pathlib import Path

from subreddit_lens import ingest_archive, load_comments, load_config

logging.basicConfig(level=logging.INFO, format="%(message)s")

# Locate the repository root so paths work regardless of the working directory.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
config = load_config(REPO_ROOT / "examples" / "litigi" / "subreddit-lens.toml")
config

## Comments

In [ ]:
n_comments = ingest_archive(
    config.comments_archive,
    config.comments_parquet,
    kind="comments",
    extra_columns=["permalink"],  # used by 05_nlp.ipynb
    condition=config.date_filter(),
)
n_comments

## Submissions (optional)

In [ ]:
if config.submissions_archive.exists():
    n_submissions = ingest_archive(
        config.submissions_archive,
        config.submissions_parquet,
        kind="submissions",
        condition=config.date_filter(),
    )
else:
    print(f"{config.submissions_archive} not found: skipping submissions.")

## Check

In [ ]:
comments = load_comments(config.comments_parquet)
print(f"{len(comments):,} comments from {comments['author'].nunique():,} authors")
print(f"{comments['created_dt'].min()} -> {comments['created_dt'].max()}")
comments.head()